# Inference Demo

Implementing Frequentist and Bayesian inference.

Before we begin...

In [ ]:
# we'll only be using numpy, scipy
# and matplotlib for plotting
import numpy as np 
import scipy as sp
%matplotlib inline
import matplotlib.pyplot as plt

In [ ]:
# set the random seed so we can reproduce everything
np.random.seed(1) 

## Example 0: Simple Photon Counts
Let's revisit the example from the lecture. 

> Imagine a hypothetical simplistic example, where we point our telescope to observe the light from a single star with true flux $F_{\rm true}$, which remains constant with time. We will ignore any systematic errors. We perform $N$ measurements with our telescope, where the $i^{\rm th}$ measurement report an observed flux $F_i$ with uncertainty/error $e_i$.

Let's generate some "synthetic observations" for this example. We'll choose the following: 

In [ ]:
F_true = 1000 # true flux
N = 50 # measurements

In [ ]:
# flux measurements sampled from a poisson distribution
F = np.random.poisson(lam=F_true, size=N) 
e = np.sqrt(F) # uncertainties 

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ax.errorbar(F, np.arange(N), xerr=e, fmt='ok', ecolor='gray', alpha=0.5)
ax.vlines([F_true], 0, N, linewidth=5, alpha=0.2)
ax.set_xlabel("Flux")
ax.set_ylabel("measurement number")
plt.show()

Now, given this set of independent measurements $\{F_i, e_i\}$, **what is our best estimate of $F_{\rm true}$**? 

Lets start with the 
## Frequentist Approach

For a **single observation** $(F_i, e_i)$, we can write down the probability distrubtion, or likelihood, of the measurement given the true flux as:

$$P(F_i, e_i\,|\,F_{\rm true}) = \frac{1}{\sqrt{2\pi e_i^2}}\,\exp \left(-\frac{(F_i - F_{\rm true})^2}{2e_i^2}\right).$$ 

*Here we're making the reasonable assumption that the errors are Gaussian.*

Then, for all of our observations $\{(F_i, e_i)\}$, the probability distribution of the measurements given the true flux is

$$P(\{(F_i, e_i)\}\,|\,F_{\rm true}) = \prod\limits_{i=1}^{N} P(F_i, e_i\,|\,F_{\rm true}).$$

This is the "likelihood". 

For practical purposes, we often compute the log-likelihood, as likelihood values can be very small
$$\log P(\{(F_i, e_i)\}\,|\,F_{\rm true}) = \sum\limits_{i=1}^{N} \log P(F_i, e_i\,|\,F_{\rm true}) = -\frac{1}{2} \sum\limits_{i=1}^{N} \left[\log(2\pi e_i^2) + \frac{(F_i - F_{\rm true})^2}{e_i^2}\right].$$

In [ ]:
def log_likelihood(theta, F, e):
    return -0.5 * np.sum(np.log(2 * np.pi * e ** 2) + 
                         (F - theta) ** 2 / e ** 2)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
ax.plot(np.linspace(900, 1100, 1000), 
        [10**log_likelihood(_F_true, F, e) 
         for _F_true in np.linspace(900, 1100, 1000)])
ax.axvline(F_true, color='k', linestyle='--')
ax.set_xlabel(r"$F_{\rm true}$")
ax.set_xlim(975, 1025)
ax.set_ylabel(r"likelihood $P(\{(F_i, e_i)\}\,|\,F_{\rm true})$")
plt.show()

Next, lets consider the
## Bayesian approach

We start by applying Bayes' Theorem to our example: 
$$P(F_{\rm true}\,|\,\{ F_i, e_i\} ) = \frac{P(\{ F_i, e_i\} \,|\,F_{\rm true})~P(F_{\rm true})}{P(\{ F_i, e_i\})}$$

- $P(\{ F_i, e_i\} \,|\,F_{\rm true})$ is the likelihood, same as in the Frequentist approach

- $P(F_{\rm true})$ is the prior

- $P(\{ F_i, e_i\})$ is the data probability or evidence. Note that this term does not depend on $F_{\rm true}$ so it can be considered a simple normalization term.

The main practical difference from the Frequentist approach is the prior. If we set the prior $P(F_{\rm true}) \propto 1$, a uniform "noninformative" prior

In [ ]:
def log_prior(theta): 
    # uniform "noninformative" prior 
    return 1

def log_posterior(theta, F, e):
    # posterior with uniform prior and same likelihood
    return log_prior(theta) + log_likelihood(theta, F, e)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
ax.plot(np.linspace(900, 1100, 1000), [10**log_posterior(_F_true, F, e) 
         for _F_true in np.linspace(900, 1100, 1000)])
ax.axvline(F_true, color='k', linestyle='--')
ax.set_xlabel(r"$F_{\rm true}$")
ax.set_xlim(975, 1025)
ax.set_ylabel(r"$P(\{(F_i, e_i)\}\,|\,F_{\rm true})$")
plt.show()

We get the exact same results as the Frequentist approach. 

This is why
> The Frequentist approach can often be viewed as simply a special case of the Bayesian approach for some (implicit) choice of the prior.

Instead of the uniforn "non-informative" prior, let's we have the measurement:
$$\hat{F}_{\rm true} = 1002 \pm 4$$ 
from a previous experiment with a different measurement strategy and we want to include this into the computation through the prior. This is often done in cosmological parameter estimation.

In this case we'd have an informative prior: 
$$P(F_{\rm true}) = \frac{1}{\sqrt{32\pi}}\,\exp \left(-\frac{(1002 - F_{\rm true})^2}{32}\right)$$

In [ ]:
def log_prior(theta): 
    # informative prior from different measurement strategy
    return -np.sqrt(32*np.pi) - (1002. - theta)**2/32

def log_posterior(theta, F, e):
    # different prior, same likelihood as before
    return log_prior(theta) + log_likelihood(theta, F, e)

In [ ]:
fig, ax = plt.subplots(figsize=(4,3))
ax.plot(np.linspace(900, 1100, 1000), [10**log_posterior(_F_true, F, e) 
         for _F_true in np.linspace(900, 1100, 1000)])
ax.axvline(F_true, color='k', linestyle='--')
ax.set_xlabel(r"$F_{\rm true}$")
ax.set_xlim(975, 1025)
ax.set_ylabel(r"$P(\{(F_i, e_i)\}\,|\,F_{\rm true})$")
plt.show()

With the prior from the previous experiment, we get tighter posterior constraints on $F_{\rm true}$.

# Example 1: Less Simple Photon Count
Next, lets consider a slightly less simple scenario. The flux from the star is not constant. There is some *intrinsic* variability: 
$$F_{\rm true} \sim \mathcal{N}(\mu, \sigma)$$
mean flux of $\mu$ with a standard deviation of variability $\sigma$. Now, we're interested in measuring $\mu$ and $\sigma$.

Let's again generate some synthetic observations

In [ ]:
N = 100  # we'll use more measurements
mu_true, sigma_true = 1000, 15 

In [ ]:
# true flux from sampling the Gaussian
F_true = sp.stats.norm(mu_true, sigma_true).rvs(N)  
# observed flux from sampling a poisson distirbution 
F = sp.stats.poisson(F_true).rvs()
e = np.sqrt(F)  # poisson errors

In [ ]:
fig, ax = plt.subplots(figsize=(4,4))
ax.errorbar(F, np.arange(N), xerr=e, fmt='ok', ecolor='gray', alpha=0.5)
ax.set_xlabel("Flux")
ax.set_ylabel("measurement number")
plt.show()

In this scenario, we can write down the likelihood for a single observation as 
$$P(F_i, e_i\,|\,\mu, \sigma) = \frac{1}{\sqrt{2\pi (\sigma^2 + e_i^2)}}\,\exp \left(-\frac{(F_i - \mu)^2}{2(\sigma^2+e_i^2)}\right).$$ 

For all of our observations $\{(F_i, e_i)\}$, the likelihood is again:
$$P(\{(F_i, e_i)\}\,|\,F_{\rm true}) = \prod\limits_{i=1}^{N} P(F_i, e_i\,|\,F_{\rm true}).$$

In [ ]:
def log_likelihood(theta, F, e):
    return -0.5 * np.sum(np.log(2 * np.pi * (theta[1] ** 2 + e ** 2)) + 
                         (F - theta[0]) ** 2 / (theta[1] ** 2 + e ** 2))

Lets visualize what the likelihood looks like for different values of $\mu$ and $\sigma$

In [ ]:
# grid of mu and sigma
mu_grid = np.linspace(900, 1100, 100)
sig_grid = np.linspace(0.01, 50, 100)
mu_grid, sig_grid = np.meshgrid(mu_grid, sig_grid)
musig_grid = np.vstack((mu_grid.ravel(), sig_grid.ravel()))

# evaluate the log-likelihood over the grid
func_vals = np.array([log_likelihood(_musig, F, e) 
                      for _musig in musig_grid.T])

In [ ]:
fig = plt.figure(figsize=(4,3))
ax = fig.gca()
ax.pcolormesh(mu_grid, sig_grid, func_vals.reshape(mu_grid.shape), 
              shading='nearest', cmap='jet')
# true values
ax.plot(mu_true, sigma_true, marker='x', zorder=10, color='black')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\sigma$')  
fig.tight_layout()
plt.show()

The colors represent the log likelihood. You can see how it varies with different $\mu$ and $\sigma$.

Typically in a Frequentist approach, you're interested in the **maximum likelihood estimate** (MLE). We can use `scipy.optimize` to get the MLE. 

In [ ]:
# maximizing likelihood is equivalent to 
# minimizing negative log likelihood
def neg_log_likelihood(theta, F, e):
    return -log_likelihood(theta, F, e)

theta_guess = [900, 20] # initial guess 
theta_est = sp.optimize.fmin(neg_log_likelihood, 
                             theta_guess, 
                             args=(F, e))
print("""
      Maximum likelihood estimate for {0} data points:
          mu={theta[0]:.0f}, sigma={theta[1]:.0f}
      """.format(N, theta=theta_est))

Now let's say there was a prior experiment that measured $\sigma = 12 \pm 2$. We can then incorporate this as our prior in the Bayesian approach

In [ ]:
def log_prior(theta):
    # sigma needs to be positive.
    if theta[1] <= 0:
        return -np.inf
    else:
        return -(12 - theta[1])**2/8

def log_posterior(theta, F, e):
    return log_prior(theta) + log_likelihood(theta, F, e)

Let's visualize the posterior

In [ ]:
# grid of mu and sigma
mu_grid = np.linspace(900, 1100, 100)
sig_grid = np.linspace(0.01, 50, 100)
mu_grid, sig_grid = np.meshgrid(mu_grid, sig_grid)
musig_grid = np.vstack((mu_grid.ravel(), sig_grid.ravel()))

# evaluate the log-posteiror over (mu,sig) grid
func_vals = np.array([log_posterior(_musig, F, e) 
                      for _musig in musig_grid.T])

In [ ]:
fig = plt.figure(figsize=(4,3))
ax = fig.gca()
ax.pcolormesh(mu_grid, sig_grid, func_vals.reshape(mu_grid.shape), 
              shading='nearest', cmap='jet')
# true values
ax.plot(mu_true, sigma_true, marker='x', zorder=10, color='black')
ax.set_xlabel(r'$\mu$')
ax.set_ylabel(r'$\sigma$')  
fig.tight_layout()
plt.show()

In [ ]:
# again, maximize likelihood = minimize negative likelihood
def neg_log_posterior(theta, F, e):
    return -log_posterior(theta, F, e)

theta_guess = [900, 20] # initial guess 
theta_est = sp.optimize.fmin(neg_log_posterior, theta_guess, args=(F, e))
print("""
      Maximum A Posteriori (MAP) estimate for {0} data points:
          mu={theta[0]:.0f}, sigma={theta[1]:.0f}
      """.format(N, theta=theta_est))

Not bad! Recall, `mu_true, sigma_true = 1000, 15`